# Intermediate Role Mapping

This notebook implements canonical role mapping using a hybrid approach:

1. **Dictionary/Alias Match** - Fast exact and fuzzy dictionary lookup
2. **Regex Pattern Match** - Rule-based pattern matching for common role formats
3. **NLP/LLM Fallback** - Optional semantic matching for unresolved low-confidence records

## Architecture

**Input**: `workspace.silver.silver_jobs_current`  
**Output**: `workspace.intermediate.inter_job_role_map`  
**Review Queue**: `workspace.silver.silver_intermediate_review_queue`  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.role_mapping_batches`
- Automatically processes all unprocessed batches
- Idempotent: safe to re-run
- Confidence scoring at each stage

## Features
- Explicit timeout handling for LLM calls
- Audit trail for all mappings
- Performance metrics tracking

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")

batch_id = dbutils.widgets.get("batch_id").strip()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType, IntegerType
from datetime import datetime
import re
from typing import Optional, Tuple

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
INTERMEDIATE_SCHEMA = f"{CATALOG}.intermediate"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Configuration
CONFIG = {
    "input_table": f"{SILVER_SCHEMA}.silver_jobs_current",
    "output_table": f"{INTERMEDIATE_SCHEMA}.inter_job_role_map",
    "review_queue_table": f"{SILVER_SCHEMA}.silver_intermediate_review_queue",
    "confidence_threshold": 0.65,  # Lowered to 0.65 to handle German titles with extra text
    "llm_timeout_seconds": 5,
    "enable_llm_fallback": False,  # Disabled for now (pandas UDF compatibility)
}

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

In [0]:
# Expanded role dictionary with German equivalents and broader coverage
# NOW INCLUDES: Marketing, Sales, Finance (German), HR, Operations, Creative, Students

from pyspark.sql.types import StructType, StructField, StringType

print("Loading expanded role taxonomy (55+ roles, German + English)...")

expanded_roles_data = [
    # Software & Engineering
    ("SWE_SENIOR", "Senior Software Engineer", "ENG", "TECH", "senior", "senior software engineer|sr swe|lead developer|senior dev|senior entwickler|leitender entwickler"),
    ("SWE_MID", "Software Engineer", "ENG", "TECH", "mid", "software engineer|swe|software dev|developer|programmer|entwickler|softwareentwickler"),
    ("SWE_JUNIOR", "Junior Software Engineer", "ENG", "TECH", "junior", "junior software engineer|jr swe|associate developer|junior entwickler"),
    ("FRONTEND_DEV", "Frontend Developer", "ENG", "TECH", "mid", "frontend developer|frontend engineer|ui developer|frontend entwickler"),
    ("BACKEND_DEV", "Backend Developer", "ENG", "TECH", "mid", "backend developer|backend engineer|backend entwickler"),
    ("PYTHON_DEV", "Python Developer", "ENG", "TECH", "mid", "python developer|python engineer|python entwickler"),
    
    # DevOps & Infrastructure
    ("DEVOPS", "DevOps Engineer", "ITOPS", "TECH", "mid", "devops engineer|site reliability engineer|sre|platform engineer|devsecops|devsecops engineer"),
    ("SYSADMIN", "Systems Administrator", "ITOPS", "TECH", "mid", "systems administrator|sysadmin|it administrator|systemadministrator|linux administrator|linux systemadministrator"),
    ("IT_ADMIN", "IT Administrator", "ITOPS", "TECH", "mid", "it administrator|it admin|administrator|it-administrator"),
    
    # Data & Analytics
    ("DATA_SCI", "Data Scientist", "DATA", "TECH", "mid", "data scientist|ml engineer|machine learning engineer|datenwissenschaftler"),
    ("DATA_SCI_SENIOR", "Senior Data Scientist", "DATA", "TECH", "senior", "senior data scientist|lead data scientist|principal data scientist"),
    ("DATA_ANALYST", "Data Analyst", "DATA", "TECH", "mid", "data analyst|business analyst|analytics specialist|datenanalyst|consumer analytics"),
    ("DATA_ENG", "Data Engineer", "DATA", "TECH", "mid", "data engineer|big data engineer|dateningenieur"),
    
    # Product Management
    ("PM_SENIOR", "Senior Product Manager", "PRODUCT", "TECH", "senior", "senior product manager|lead product manager|principal pm|senior product owner"),
    ("PM_MID", "Product Manager", "PRODUCT", "TECH", "mid", "product manager|product owner|pm|produktmanager|business development product|ceo office product"),
    
    # Marketing & Content
    ("PERF_MARKETING", "Performance Marketing Manager", "MARKETING", "TECH", "mid", "performance marketing manager|growth marketer|performance marketer|sea manager|sem manager"),
    ("CONTENT_CREATOR", "Content Creator", "MARKETING", "MEDIA", "mid", "content creator|content specialist|content manager|ugc creator|content creation"),
    ("SOCIAL_MEDIA", "Social Media Manager", "MARKETING", "MEDIA", "mid", "social media manager|social media specialist|community manager"),
    
    # Sales & Business Development
    ("SALES_HEAD", "Head of Sales", "SALES", "BIZ", "senior", "head of sales|sales director|vertriebsleiter|vertriebsleiterin|sales lead"),
    ("SALES_MGR", "Sales Manager", "SALES", "BIZ", "mid", "sales manager|account manager|vertriebsmanager|service manager mobilfunk"),
    ("INSIDE_SALES", "Inside Sales Representative", "SALES", "BIZ", "mid", "inside sales|sales rep|inside sales contractor"),
    
    # Customer Success
    ("CSM", "Customer Success Manager", "CS", "BIZ", "mid", "customer success manager|csm|customer success"),
    ("CSM_JUNIOR", "Junior Customer Success Manager", "CS", "BIZ", "junior", "junior customer success|junior csm"),
    
    # Finance & Accounting
    ("CFO", "Chief Financial Officer", "FIN_CORP", "FIN", "executive", "cfo|chief financial officer|vp finance|group cfo|kaufmännischer geschäftsführer"),
    ("ACCOUNTANT", "Accountant", "FIN_ACCT", "FIN", "mid", "accountant|staff accountant|accounting specialist|buchhalter|bilanzbuchhalter"),
    ("FIN_ANALYST", "Financial Analyst", "FIN_CORP", "FIN", "mid", "financial analyst|finance analyst|corporate finance analyst|financial controlling analyst"),
    ("TAX_ADVISOR", "Tax Advisor", "FIN_ACCT", "FIN", "senior", "tax advisor|steuerberater|tax consultant|steuerberaterin"),
    ("AR_SPECIALIST", "Accounts Receivable Specialist", "FIN_ACCT", "FIN", "mid", "accounts receivable|ar specialist|debitorenbuchhalter"),
    ("FINANCE_HEAD", "Head of Finance", "FIN_CORP", "FIN", "senior", "head of finance|finance director|leiter finanzbuchhaltung|leiter rechnungswesen"),
    
    # Operations & Management
    ("COO", "Chief Operating Officer", "OPS", "BIZ", "executive", "coo|chief operating officer|head of operations|chief operating sales officer"),
    ("OPS_MGR", "Operations Manager", "OPS", "BIZ", "mid", "operations manager|ops manager|bereichsleitung|real estate operations"),
    ("PROJECT_MGR", "Project Manager", "OPS", "BIZ", "mid", "project manager|projektmanager|projektleitung|projektmanagement"),
    
    # HR & People
    ("HR_MGR", "HR Manager", "HR", "BIZ", "mid", "hr manager|human resources manager|hr lead|personalmanager"),
    ("HR_GEN", "HR Generalist", "HR", "BIZ", "mid", "hr generalist|hr specialist|hr business partner"),
    
    # Creative & Entertainment
    ("CREATIVE", "Creative Specialist", "CREATIVE", "MEDIA", "mid", "creative expert|creative specialist|kreativexperte"),
    ("ARTIST_MGR", "Artist Manager", "ENTERTAINMENT", "MEDIA", "mid", "artist manager|talent manager|influencer manager"),
    
    # Administrative & Support
    ("DATA_ENTRY", "Data Entry Specialist", "ADMIN", "BIZ", "junior", "data entry|datenerfassung|mitarbeiter datenerfassung"),
    
    # Specialized Roles
    ("PROCUREMENT", "Procurement Manager", "OPS", "BIZ", "mid", "procurement manager|vergabemanager|vergabemanagerin|purchasing manager"),
    ("SERVICENOW", "ServiceNow Consultant", "ITOPS", "TECH", "mid", "servicenow consultant|servicenow presales|servicenow specialist"),
    ("CYBERSEC", "Cybersecurity Manager", "SECURITY", "TECH", "senior", "cybersecurity manager|bereichsleitung cybersecurity|security governance|quality governance"),
    
    # Interns & Students
    ("WERKSTUDENT", "Working Student", "INTERN", "GEN", "junior", "werkstudent|werkstudentin|working student|student assistant"),
    ("TRAINEE", "Trainee", "INTERN", "GEN", "junior", "trainee|praktikant|intern"),
    
    # Leadership & Directors
    ("HEAD_DIGITAL", "Head of Digital", "TECH_LEADERSHIP", "TECH", "senior", "head of digital|head of systems|digital director"),
    ("TEST_LAB_HEAD", "Head of Test Lab", "ENG_LEADERSHIP", "TECH", "senior", "head of test lab|test lab manager|qa director|global engineering verification"),
]

schema = StructType([
    StructField("role_key", StringType(), True),
    StructField("canonical_role", StringType(), True),
    StructField("family_key", StringType(), True),
    StructField("sector_key", StringType(), True),
    StructField("seniority", StringType(), True),
    StructField("aliases", StringType(), True)
])

roles_raw_df = spark.createDataFrame(expanded_roles_data, schema=schema)

# Explode aliases into individual dictionary entries
role_dict_df = roles_raw_df.select(
    F.col("canonical_role"),
    F.col("family_key"),
    F.col("sector_key"),
    F.col("seniority"),
    F.explode(F.split(F.col("aliases"), "\\|")).alias("alias")
).withColumn(
    "alias", F.lower(F.trim(F.col("alias")))
).withColumn(
    "match_type", 
    F.when(F.col("alias") == F.lower(F.col("canonical_role")), "exact").otherwise("alias")
).withColumn(
    "confidence_boost",
    F.when(F.col("match_type") == "exact", 1.0).otherwise(0.9)
)

print(f"✅ Loaded {roles_raw_df.count()} canonical roles with {role_dict_df.count()} total aliases")

In [0]:
# Create metadata table to track role-mapped batches
metadata_table = f"{METADATA_SCHEMA}.role_mapping_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  jobs_processed INT,
  canonical_count INT,
  review_queue_count INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been role-mapped'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("jobs_processed", IntegerType(), True),
    StructField("canonical_count", IntegerType(), True),
    StructField("review_queue_count", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# OPTIONAL: Uncomment to reset and reprocess all batches with new logic
# WARNING: This will delete all existing role mappings and metadata

RESET_AND_REPROCESS = True  # Set to True to clear and reprocess

if RESET_AND_REPROCESS:
    print("⚠️  RESETTING: Deleting all role mappings and metadata...")
    
    # Clear output table
    spark.sql(f"DELETE FROM {CONFIG['output_table']}")
    
    # Clear review queue
    spark.sql(f"DELETE FROM {CONFIG['review_queue_table']} WHERE status = 'pending'")
    
    # Clear metadata
    spark.sql(f"DELETE FROM {metadata_table}")
    
    print("✅ Reset complete. All batches will be reprocessed with new expanded dictionary and improved fuzzy matching.")
else:
    print("ℹ️  Using incremental mode (only new batches). Set RESET_AND_REPROCESS=True to reprocess everything.")

In [0]:
# Build lookup dictionaries for fast matching
role_dict_lookup = {}
for row in role_dict_df.collect():
    alias_key = row["alias"].lower().strip()
    role_dict_lookup[alias_key] = {
        "canonical_role": row["canonical_role"],
        "confidence": row["confidence_boost"],
        "method": "dictionary"
    }

def match_dictionary(role_text: str) -> Optional[Tuple[str, float, str]]:
    """Improved matching with better fuzzy logic. Returns (canonical_role, confidence, method) or None."""
    if not role_text:
        return None
    
    role_key = role_text.lower().strip()
    
    # Remove German gender notation for better matching
    role_key_clean = role_key.replace("(m/w/d)", "").replace("(m/f/d)", "").replace("(gn)", "").strip()
    
    # Stage 1: Exact match (highest confidence)
    if role_key_clean in role_dict_lookup:
        result = role_dict_lookup[role_key_clean]
        return (result["canonical_role"], result["confidence"], result["method"])
    
    # Stage 2: Improved fuzzy match with stricter rules
    best_match = None
    best_score = 0.0
    
    for alias, mapping in role_dict_lookup.items():
        # Skip very short aliases to avoid false positives
        if len(alias) < 4:
            continue
            
        # Check if alias is a significant substring
        if alias in role_key_clean:
            # Calculate match quality based on length ratio
            match_ratio = len(alias) / len(role_key_clean)
            
            # Only accept if alias covers at least 20% of the input
            if match_ratio >= 0.2:
                confidence = mapping["confidence"] * 0.95 * min(match_ratio * 1.5, 1.0)
                
                if confidence > best_score:
                    best_match = (mapping["canonical_role"], confidence, "dictionary_fuzzy")
                    best_score = confidence
        
        # Check reverse (input is substring of alias) - only for longer inputs
        elif role_key_clean in alias and len(role_key_clean) >= 5:
            match_ratio = len(role_key_clean) / len(alias)
            
            if match_ratio >= 0.4:  # Input must be at least 40% of the alias
                confidence = mapping["confidence"] * 0.90
                
                if confidence > best_score:
                    best_match = (mapping["canonical_role"], confidence, "dictionary_fuzzy")
                    best_score = confidence
    
    return best_match

In [0]:
# Define regex patterns for common role structures
ROLE_PATTERNS = [
    # Pattern: (regex, canonical_role, confidence)
    (r"(?i)^(senior|sr|lead)\s*software\s*(engineer|dev)", "Senior Software Engineer", 0.9),
    (r"(?i)^(junior|jr)\s*software\s*(engineer|dev)", "Junior Software Engineer", 0.9),
    (r"(?i)^software\s*(engineer|dev)", "Software Engineer", 0.85),
    (r"(?i)^(senior|sr|lead)\s*data\s*scientist", "Senior Data Scientist", 0.9),
    (r"(?i)^data\s*scientist", "Data Scientist", 0.85),
    (r"(?i)^(senior|sr)\s*product\s*manager", "Senior Product Manager", 0.9),
    (r"(?i)^product\s*manager", "Product Manager", 0.85),
    (r"(?i)^(chief|head\s+of)\s*technology", "Chief Technology Officer", 0.9),
    (r"(?i)^engineering\s*manager", "Engineering Manager", 0.85),
]

def match_regex(role_text: str) -> Optional[Tuple[str, float, str]]:
    """Match role against regex patterns. Returns (canonical_role, confidence, method) or None."""
    if not role_text:
        return None
    
    for pattern, canonical, confidence in ROLE_PATTERNS:
        if re.match(pattern, role_text):
            return (canonical, confidence, "regex")
    
    return None

In [0]:
def match_llm(role_text: str, timeout_seconds: int = 5) -> Optional[Tuple[str, float, str]]:
    """Match role using Databricks AI with German translation. Returns (canonical_role, confidence, method) or None."""
    if not role_text:
        return None
    
    try:
        # Get list of canonical roles for LLM classification
        canonical_roles_list = [row.canonical_role for row in roles_raw_df.select("canonical_role").distinct().collect()]
        
        # Build prompt for LLM
        prompt = f"""You are a job title classifier. Given this job title, identify the best matching canonical role.

Job Title: {role_text}

Canonical Roles:
{', '.join(canonical_roles_list[:20])}  # Use top 20 most common roles

If the job title is in German, translate it to English first.
Return ONLY the canonical role name that best matches, or "UNKNOWN" if no good match.
"""
        
        # Use Databricks AI query (if available)
        try:
            result = spark.sql(f"""
                SELECT ai_query(
                    'databricks-meta-llama-3-1-70b-instruct',
                    '{prompt.replace("'", "''")}'
                ) as response
            """).first()
            
            if result and result.response:
                suggested_role = result.response.strip()
                
                # Check if suggested role is in our canonical list
                if suggested_role in canonical_roles_list:
                    return (suggested_role, 0.65, "llm")
        except:
            # AI function not available or failed, skip
            pass
        
        return None
        
    except Exception as e:
        # Silently fail - LLM is optional fallback
        return None

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd
import re as re_module

# Broadcast dictionary for worker access
role_dict_broadcast = {alias: mapping for alias, mapping in role_dict_lookup.items()}
confidence_threshold_val = CONFIG["confidence_threshold"]
enable_llm_val = CONFIG["enable_llm_fallback"]

@pandas_udf("struct<canonical_role:string,confidence:double,method:string>")
def hybrid_match_udf(titles: pd.Series) -> pd.DataFrame:
    """Pandas UDF for hybrid role matching (Spark Connect compatible)."""
    
    def match_single_role(role_text):
        """Match a single role title."""
        if pd.isna(role_text) or not role_text:
            return {"canonical_role": None, "confidence": None, "method": None}
        
        role_key = str(role_text).lower().strip()
        role_key_clean = role_key.replace("(m/w/d)", "").replace("(m/f/d)", "").replace("(gn)", "").strip()
        
        # Stage 1: Exact dictionary match
        if role_key_clean in role_dict_broadcast:
            result = role_dict_broadcast[role_key_clean]
            return {
                "canonical_role": result["canonical_role"],
                "confidence": result["confidence"],
                "method": result["method"]
            }
        
        # Stage 2: Fuzzy dictionary match
        best_match = None
        best_score = 0.0
        
        for alias, mapping in role_dict_broadcast.items():
            if len(alias) < 4:
                continue
            
            if alias in role_key_clean:
                match_ratio = len(alias) / len(role_key_clean)
                if match_ratio >= 0.2:  # Lowered from 0.3
                    confidence = mapping["confidence"] * 0.95 * min(match_ratio * 1.5, 1.0)
                    if confidence > best_score:
                        best_match = {
                            "canonical_role": mapping["canonical_role"],
                            "confidence": confidence,
                            "method": "dictionary_fuzzy"
                        }
                        best_score = confidence
            
            elif role_key_clean in alias and len(role_key_clean) >= 5:
                match_ratio = len(role_key_clean) / len(alias)
                if match_ratio >= 0.4:  # Lowered from 0.5
                    confidence = mapping["confidence"] * 0.90
                    if confidence > best_score:
                        best_match = {
                            "canonical_role": mapping["canonical_role"],
                            "confidence": confidence,
                            "method": "dictionary_fuzzy"
                        }
                        best_score = confidence
        
        if best_match:
            return best_match
        
        # Stage 3: Regex patterns
        patterns = [
            (r"(?i)^(senior|sr|lead)\s*software\s*(engineer|dev)", "Senior Software Engineer", 0.9),
            (r"(?i)^(junior|jr)\s*software\s*(engineer|dev)", "Junior Software Engineer", 0.9),
            (r"(?i)^software\s*(engineer|dev)", "Software Engineer", 0.85),
            (r"(?i)^(senior|sr|lead)\s*data\s*scientist", "Senior Data Scientist", 0.9),
            (r"(?i)^data\s*scientist", "Data Scientist", 0.85),
            (r"(?i)^(senior|sr)\s*product\s*manager", "Senior Product Manager", 0.9),
            (r"(?i)^product\s*manager", "Product Manager", 0.85),
        ]
        
        for pattern, canonical, conf in patterns:
            if re_module.match(pattern, role_text):
                return {"canonical_role": canonical, "confidence": conf, "method": "regex"}
        
        # No match found
        return {"canonical_role": None, "confidence": None, "method": None}
    
    # Apply matching to all rows in the batch
    results = titles.apply(match_single_role)
    return pd.DataFrame(results.tolist())

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in current table
    all_batches_query = f"""
        SELECT DISTINCT current_batch_id as batch_id, source_name
        FROM {CONFIG["input_table"]}
        WHERE is_active = true AND soft_delete_flag = false
        AND current_batch_id IS NOT NULL
        AND current_batch_id != ''
        AND source_name IS NOT NULL
        AND title_normalized IS NOT NULL
    """
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already processed batches
    processed_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unprocessed batches (in current but not in metadata)
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unprocessed batches", "total_mapped": 0})
    
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
# Initialize tracking
total_jobs_processed = 0
total_canonical = 0
total_review_queue = 0
processed_batch_count = 0
failed_batches = []

confidence_threshold = CONFIG["confidence_threshold"]

# Process each batch
for batch_id_val, source_name_val in batches_to_process:
    print(f"\nProcessing role mapping for batch {batch_id_val} ({source_name_val})...", end=" ")
    
    try:
        # Load jobs for this batch
        batch_jobs_df = spark.sql(f"""
            SELECT 
                enterprise_job_id,
                title_normalized,
                source_name,
                current_batch_id
            FROM {CONFIG["input_table"]}
            WHERE current_batch_id = '{batch_id_val}'
            AND source_name = '{source_name_val}'
            AND is_active = true
            AND soft_delete_flag = false
            AND title_normalized IS NOT NULL
        """)
        
        batch_count = batch_jobs_df.count()
        
        if batch_count == 0:
            print("⚠ No jobs found (skipped)")
            continue
        
        # Apply hybrid matching
        matched_df = batch_jobs_df.withColumn(
            "match_result",
            hybrid_match_udf(F.col("title_normalized"))
        ).select(
            F.col("enterprise_job_id"),
            F.col("title_normalized").alias("standardized_role"),
            F.col("match_result.canonical_role").alias("canonical_role"),
            F.col("match_result.confidence").alias("confidence"),
            F.col("match_result.method").alias("matching_method"),
            F.lit(batch_id_val).alias("batch_id"),
            F.lit(source_name_val).alias("source_name")
        )
        
        # Split into canonical roles (high confidence) and review queue (low confidence or no match)
        canonical_df = matched_df.filter(
            (F.col("canonical_role").isNotNull()) & 
            (F.col("confidence") >= confidence_threshold)
        )
        
        review_queue_df = matched_df.filter(
            (F.col("canonical_role").isNull()) | 
            (F.col("confidence") < confidence_threshold)
        )
        
        canonical_count = canonical_df.count()
        review_count = review_queue_df.count()
        
        # Write canonical roles to intermediate.inter_job_role_map
        if canonical_count > 0:
            output_df = canonical_df.select(
                F.md5(F.concat(F.col("enterprise_job_id"), F.col("canonical_role"))).alias("role_map_id"),
                F.col("enterprise_job_id"),
                F.col("standardized_role").alias("title_normalized"),
                F.col("canonical_role").alias("canonical_role_id"),
                F.lit("Technology").alias("role_family"),
                F.when(F.col("canonical_role").contains("Senior"), "SENIOR")
                 .when(F.col("canonical_role").contains("Lead"), "SENIOR")
                 .when(F.col("canonical_role").contains("Junior"), "ENTRY")
                 .when(F.col("canonical_role").contains("Chief"), "EXECUTIVE")
                 .otherwise("MID").alias("seniority_level"),
                F.when(F.col("matching_method") == "dictionary", "EXACT_MATCH")
                 .when(F.col("matching_method") == "dictionary_fuzzy", "FUZZY")
                 .when(F.col("matching_method") == "regex", "EXACT_MATCH")
                 .when(F.col("matching_method") == "llm", "LLM")
                 .otherwise("MANUAL").alias("normalization_method"),
                F.col("confidence").cast("decimal(5,4)").alias("normalization_confidence"),
                F.to_json(F.struct(
                    F.col("standardized_role").alias("input"),
                    F.col("matching_method").alias("method")
                )).alias("explanation_json"),
                F.lit(batch_id_val).alias("effective_batch_id")
            )
            
            output_df.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(CONFIG["output_table"])
        
        # Write review queue
        if review_count > 0:
            review_df = review_queue_df.select(
                F.expr("uuid()").alias("review_id"),
                F.col("enterprise_job_id"),
                F.when(F.col("canonical_role").isNull(), "no_match")
                 .otherwise("low_confidence").alias("issue_type"),
                F.concat(
                    F.lit("Role: "),
                    F.col("standardized_role"),
                    F.lit(", Suggested: "),
                    F.coalesce(F.col("canonical_role"), F.lit("none"))
                ).alias("issue_detail"),
                F.coalesce(F.col("confidence"), F.lit(0.0)).cast("decimal(5,4)").alias("confidence"),
                F.current_timestamp().alias("queued_at"),
                F.lit("pending").alias("status")
            )
            
            review_df.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(CONFIG["review_queue_table"])
        
        # Record metadata
        metadata_row = [(
            batch_id_val,
            source_name_val,
            batch_count,
            canonical_count,
            review_count,
            run_timestamp_py,
            run_id,
            "success"
        )]
        
        metadata_df = spark.createDataFrame(metadata_row, schema=metadata_schema)
        metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)
        
        total_jobs_processed += batch_count
        total_canonical += canonical_count
        total_review_queue += review_count
        processed_batch_count += 1
        
        print(f"✓ Jobs:{batch_count} Canonical:{canonical_count} Review:{review_count}")
        
    except Exception as e:
        error_msg = str(e)
        print(f"✗ ERROR: {error_msg}")
        failed_batches.append((batch_id_val, source_name_val, error_msg))
        
        # Record failure in metadata
        try:
            metadata_row = [(
                batch_id_val,
                source_name_val,
                0,
                0,
                0,
                run_timestamp_py,
                run_id,
                f"failed: {error_msg[:200]}"
            )]
            metadata_df = spark.createDataFrame(metadata_row, schema=metadata_schema)
            metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)
        except Exception as meta_error:
            print(f"  Could not record failure in metadata: {meta_error}")
        
        continue

print(f"\nProcessed {processed_batch_count} batches, {total_jobs_processed} jobs, {total_canonical} canonical roles, {total_review_queue} review queue")

In [0]:
import json

# Show role mapping batch history
if processed_batch_count > 0:
    print("\nRole Mapping Batch History:")
    metadata_df = spark.sql(f"""
        SELECT batch_id, source_name, jobs_processed, canonical_count, review_queue_count, processed_at, status
        FROM {metadata_table}
        ORDER BY processed_at DESC
        LIMIT 20
    """)
    display(metadata_df)
    
    # Show role mapping summary by method
    print("\nRole Mapping Method Distribution:")
    method_summary_df = spark.sql(f"""
        SELECT 
            normalization_method,
            COUNT(*) as mapping_count,
            ROUND(AVG(normalization_confidence), 3) as avg_confidence
        FROM {CONFIG["output_table"]}
        GROUP BY normalization_method
        ORDER BY mapping_count DESC
    """)
    display(method_summary_df)
    
    # Show recent mappings
    print("\nRecent Role Mappings:")
    recent_mappings_df = spark.sql(f"""
        SELECT 
            enterprise_job_id,
            title_normalized,
            canonical_role_id,
            seniority_level,
            normalization_method,
            normalization_confidence,
            effective_batch_id
        FROM {CONFIG["output_table"]}
        ORDER BY role_map_id DESC
        LIMIT 10
    """)
    display(recent_mappings_df)

# Exit summary
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "jobs_processed": total_jobs_processed,
    "canonical_count": total_canonical,
    "review_queue_count": total_review_queue,
    "success_rate": f"{(total_canonical / total_jobs_processed * 100) if total_jobs_processed > 0 else 0:.1f}%",
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(result))